In [1]:
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


In [2]:
import librosa
from IPython import display as disp
from scipy.io import wavfile

import src.synthesizer as synthesizer
import src.synthesizer_bnf_vq as synthesizer_bnf_vq

/home/grads/q/quamer.waris/anaconda3/envs/streamVC/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
/mnt/nvme-data1/waris/PSI-TAMU/streamVC/anonymization/demo_speaker_embeddings.py:5: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier
INFO:fairseq.tasks.text_to_speech:Please install tensorboardX: pip install tensorboardX


In [3]:
config = "experiments/base/config.json"
ckpt = "experiments/base/g_01120000"
device = "cuda"

In [4]:
synthesizer.load_models(config, ckpt, device)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Loading 'experiments/base/g_01120000'
Complete.
Loaded Encoder and Generator models.


INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py

Loaded Speaker encoder.
Model type of anonymizer: gan


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding

Loaded Anonymizer.


In [23]:
wav_path = "/mnt/nvme-data1/waris/datasets/SLT_subset/arctic_a0010.wav"
ref_wav_path = "/mnt/nvme-data1/waris/datasets/SLT_subset/arctic_a0010.wav"

In [24]:
or_audio, _ = librosa.load(wav_path, sr=16000)

disp.Audio(or_audio, rate=16000)

In [7]:
wavfile.write(f'experiments/base/examples/bdl_arctic_a0010.wav', 16000, or_audio)


In [8]:
ref_audio, _ = librosa.load(ref_wav_path, sr=16000)

disp.Audio(ref_audio, rate=16000)

### Voice Conversion

In [9]:
gen_audio, rtf_, e2e_latency_ = synthesizer.process_wav_stream(wav_path, ref_wav_path, chunk_size=160)
print(f"RTF: {rtf_}, E2E Latency: {e2e_latency_}")

RTF: 0.1856492388816107, E2E Latency: 189.70387822105772


In [10]:
disp.Audio(gen_audio, rate=16000)

In [11]:
wavfile.write(f'experiments/base/examples/bdl_recon_arctic_a0010.wav', 16000, gen_audio)


### Anonymization

In [12]:
gen_audio, rtf_, e2e_latency_ = synthesizer.process_wav_stream(wav_path, chunk_size=160)
print(f"RTF: {rtf_}, E2E Latency: {e2e_latency_}")

RTF: 0.1708525986898513, E2E Latency: 187.3364157903762


In [13]:
disp.Audio(gen_audio, rate=16000)

In [14]:
wavfile.write(f'experiments/base/examples/bdl_anon_arctic_a0010.wav', 16000, gen_audio)

### VQed

In [15]:
config = "/mnt/nvme-data1/waris/PSI-TAMU/streamVC/experiments/base/bnf_vq_ft_512/config.json"
qckpt = "/mnt/nvme-data1/waris/PSI-TAMU/streamVC/experiments/base/bnf_vq_ft_512/g_00500000"
ckpt = "/mnt/nvme-data1/waris/PSI-TAMU/streamVC/experiments/base/bnf_vq_ft_512/g_00500000"

In [16]:
synthesizer_bnf_vq.load_models(config, ckpt, qckpt, device)

Loading '/mnt/nvme-data1/waris/PSI-TAMU/streamVC/experiments/base/bnf_vq_ft_512/g_00500000'
Complete.
Loading '/mnt/nvme-data1/waris/PSI-TAMU/streamVC/experiments/base/bnf_vq_ft_512/g_00500000'


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Complete.
Loaded Encoder and Generator models.


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding

Loaded Speaker encoder.
Model type of anonymizer: gan


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding

Loaded Anonymizer.


In [25]:
gen_audio, rtf_, e2e_latency_ = synthesizer_bnf_vq.process_wav_stream(wav_path, ref_wav_path, chunk_size=160)
print(f"RTF: {rtf_}, E2E Latency: {e2e_latency_}")

RTF: 0.19908155265607333, E2E Latency: 191.85304842497175


In [26]:
disp.Audio(gen_audio, rate=16000)

In [27]:
wavfile.write(f'experiments/base/examples/slt_recon_vq512_arctic_a0010.wav', 16000, gen_audio)

In [28]:
gen_audio, rtf_, e2e_latency_ = synthesizer_bnf_vq.process_wav_stream(wav_path, chunk_size=160)
print(f"RTF: {rtf_}, E2E Latency: {e2e_latency_}")

RTF: 0.1908499943582635, E2E Latency: 190.53599909732216


In [29]:
disp.Audio(gen_audio, rate=16000)

In [30]:
wavfile.write(f'experiments/base/examples/slt_anon_vq512_arctic_a0010.wav', 16000, gen_audio)